In [ ]:
# 🏥 Scenario: The Hospital’s AI Appointment Agent
# Imagine a hospital has deployed an AI-powered agent named MediLLM.
# Its job is to monitor patient appointment requests and decide whether the requested slot is available or needs rescheduling.

# Step 1 — Thinking (Reasoning)
# Whenever a patient request comes in, MediLLM first analyzes the details:
# - Example: “Request for Dr. Sharma at 10 AM tomorrow”
# - MediLLM thinks: “Analyzing appointment request: Dr. Sharma, 10 AM tomorrow.”

# Step 2 — Planning (Choosing an Action)
# Based on the thought, MediLLM decides which tool to use:
# - If the requested slot is free, it plans to confirm appointment.
# - If the slot is already booked, it plans to suggest rescheduling.

# Step 3 — Acting (Running Tools)
# MediLLM hands over the decision to AppointmentTools, which executes the action:
# - confirm_slot → returns “Appointment Confirmed”
# - suggest_reschedule → returns “Slot Unavailable, Please Reschedule”

# Step 4 — Answering (Final Decision)
# Once a clear classification is reached, MediLLM announces the decision:
# - “Decision: Appointment Confirmed”
# - “Decision: Slot Unavailable, Please Reschedule”

In [1]:
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.7/139.7 kB 7.1 MB/s eta 0:00:00


In [2]:
from google.colab import userdata
from groq import Groq

# 🔑 Load API key securely
api_key = userdata.get("GROQ_API_KEY")

# Initialize client
client = Groq(api_key=api_key)

In [7]:
# ================================
# 🏥 MediLLM - FINAL STABLE VERSION
# ================================

!pip install groq

from google.colab import userdata
from groq import Groq
import re
import json

# 🔑 Load API key from Colab Secrets
client = Groq(api_key=userdata.get("GROQ_API_KEY"))

# ================================
# 🧰 TOOLS (Step 3: Acting)
# ================================

class AppointmentTools:

    @staticmethod
    def confirm_slot():
        return "✅ Appointment Confirmed"

    @staticmethod
    def suggest_reschedule():
        return "❌ Slot Unavailable, Please Reschedule"


# ================================
# 🧠 HELPERS
# ================================

def clean_text(text):
    # Remove markdown symbols and extra spaces
    return re.sub(r'[*\-]', '', text).strip()

def normalize_time(time_str):
    time_str = time_str.lower()

    # Remove natural words
    for word in ["tomorrow", "today", "next day"]:
        time_str = time_str.replace(word, "")

    return time_str.strip().upper()

def extract_json(text):
    # Remove ```json and ```
    text = text.strip()
    text = re.sub(r"```json", "", text)
    text = re.sub(r"```", "", text)
    return text.strip()


# ================================
# 🤖 AGENT
# ================================

class MediLLM:

    def __init__(self):
        self.booked_slots = {
            "Dr. Sharma": ["10 AM", "2 PM"],
            "Dr. Mehta": ["11 AM"]
        }

    # ================================
    # Step 1: THINKING
    # ================================
    def think(self, query):
        print("\n🧠 Step 1: Thinking...")

        prompt = f"""
        Extract doctor and time in JSON format ONLY.

        Example:
        {{
          "doctor": "Dr. Sharma",
          "time": "10 AM"
        }}

        Request: {query}
        """

        response = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[{"role": "user", "content": prompt}]
        )

        output = response.choices[0].message.content
        print(output)
        return output

    # ================================
    # Step 2: PLANNING
    # ================================
    def plan(self, thought):
        print("\n📋 Step 2: Planning...")

        doctor, time = "", ""

        try:
            # 🔥 Clean JSON block
            clean_output = extract_json(thought)

            data = json.loads(clean_output)

            doctor = clean_text(data.get("doctor", ""))
            time = clean_text(data.get("time", ""))

        except Exception:
            print("⚠️ JSON parsing failed, using fallback...")
            for line in thought.split("\n"):
                if "Doctor" in line:
                    doctor = clean_text(line.split(":")[-1])
                if "Time" in line:
                    time = clean_text(line.split(":")[-1])

        # Normalize time
        time = normalize_time(time)

        print(f"👉 Doctor: {doctor}")
        print(f"👉 Time: {time}")

        # Decision logic
        if doctor in self.booked_slots and time in self.booked_slots[doctor]:
            print("📌 Plan: Slot is booked → Reschedule")
            return "reschedule", doctor, time
        else:
            print("📌 Plan: Slot is free → Confirm")
            return "confirm", doctor, time

    # ================================
    # Step 3: ACTING
    # ================================
    def act(self, action, doctor, time):
        print("\n⚙️ Step 3: Acting...")

        if action == "confirm":
            # Add booking
            if doctor not in self.booked_slots:
                self.booked_slots[doctor] = []

            self.booked_slots[doctor].append(time)
            return AppointmentTools.confirm_slot()

        return AppointmentTools.suggest_reschedule()

    # ================================
    # Step 4: ANSWERING
    # ================================
    def answer(self, result):
        print("\n📢 Step 4: Final Decision")
        print(result)
        return result

    # ================================
    # FULL PIPELINE
    # ================================
    def run(self, query):
        print("\n===============================")
        print(f"🧾 User Request: {query}")

        thought = self.think(query)
        action, doctor, time = self.plan(thought)
        result = self.act(action, doctor, time)
        return self.answer(result)


# ================================
# 🚀 TEST CASES
# ================================

agent = MediLLM()

# ❌ Already booked
agent.run("Request for Dr. Sharma at 10 AM tomorrow")

# ✅ First booking
agent.run("Book appointment with Dr. Mehta at 3 PM")

# ❌ Duplicate booking
agent.run("Book appointment with Dr. Mehta at 3 PM")


🧾 User Request: Request for Dr. Sharma at 10 AM tomorrow

🧠 Step 1: Thinking...
{
  "doctor": "Dr. Sharma",
  "time": "10 AM tomorrow"
}

📋 Step 2: Planning...
👉 Doctor: Dr. Sharma
👉 Time: 10 AM
📌 Plan: Slot is booked → Reschedule

⚙️ Step 3: Acting...

📢 Step 4: Final Decision
❌ Slot Unavailable, Please Reschedule

🧾 User Request: Book appointment with Dr. Mehta at 3 PM

🧠 Step 1: Thinking...
{
  "doctor": "Dr. Mehta",
  "time": "3 PM"
}

📋 Step 2: Planning...
👉 Doctor: Dr. Mehta
👉 Time: 3 PM
📌 Plan: Slot is free → Confirm

⚙️ Step 3: Acting...

📢 Step 4: Final Decision
✅ Appointment Confirmed

🧾 User Request: Book appointment with Dr. Mehta at 3 PM

🧠 Step 1: Thinking...
{
  "doctor": "Dr. Mehta",
  "time": "3 PM"
}

📋 Step 2: Planning...
👉 Doctor: Dr. Mehta
👉 Time: 3 PM
📌 Plan: Slot is booked → Reschedule

⚙️ Step 3: Acting...

📢 Step 4: Final Decision
❌ Slot Unavailable, Please Reschedule


'❌ Slot Unavailable, Please Reschedule'